# Image Classification
Identify the damage types present in each road image.


In [ ]:
from pathlib import Path
import sys

# Resolve shared modules and portable project settings.
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'configs/project.json').is_file()),None)
if ROOT is None:
    raise FileNotFoundError('RoadScan project configuration was not found.')
if str(ROOT/'src') not in sys.path:
    sys.path.insert(0,str(ROOT/'src'))
from roadscan.settings import load_settings
settings=load_settings(ROOT)


In [ ]:
from pathlib import Path
import collections, csv, hashlib, io, json, math, os, random, time, zipfile
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import average_precision_score, precision_recall_fscore_support
from IPython.display import display

IS_COLAB = Path('/content').exists() and os.name != 'nt'
IS_KAGGLE = Path('/kaggle/input').exists()
DATA_SOURCE = settings['dataset_source']
OUT_DIR = ROOT / 'results' / 'baseline_resnet18'

EPOCHS = 13                 # TOTAL target; completed runs are loaded, never repeated.
BATCH_SIZE = 32
IMG_SIZE = 224             # Standard ResNet18 input; fixed before evaluation.
LEARNING_RATE = 0.001
SEED = 42
MODE = 'train_or_load'     # 'load' evaluates saved weights without training.
PRETRAINED = True          # Same ImageNet starting point as the original notebook.
IMAGENET_WEIGHTS_PATH = '' # Optional local resnet18-f37072fd.pth; blank downloads it.
MAIN_MODEL_PREDICTIONS_DIR = '' # Optional directory with matching prediction CSVs + protocol.json.

CLASS_NAMES = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole']
NUM_CLASSES = 4
ALL_SPLITS = ['train', 'val', 'test', 'saudi_test']
TEST_SPLITS = ['test', 'saudi_test']
THRESHOLD = 0.5
assert 1 <= EPOCHS <= 13 and MODE in {'train_or_load', 'load'}
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| Total epoch target:', EPOCHS)


In [ ]:
if IS_COLAB and str(DATA_SOURCE).startswith('/content/drive/'):
    from google.colab import drive
    drive.mount('/content/drive')
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

DATA_SOURCE = Path(DATA_SOURCE).expanduser()
if not DATA_SOURCE.exists():
    search_root = Path('/kaggle/input') if IS_KAGGLE else Path('/content') if IS_COLAB else DATA_SOURCE.parent
    found = sorted(search_root.rglob('RoadScan_Dataset.zip')) if search_root.exists() else []
    if len(found) == 1:
        DATA_SOURCE = found[0]
    else:
        raise FileNotFoundError(f'Set DATA_SOURCE to the final ZIP or extracted dataset directory. Found: {found}')

if DATA_SOURCE.is_file():
    assert DATA_SOURCE.suffix.lower() == '.zip', 'Expected a ZIP or an extracted folder.'
    archive_hash = file_sha256(DATA_SOURCE)
    base = Path('/kaggle/working/data') if IS_KAGGLE else Path('/content/data') if IS_COLAB else OUT_DIR / 'data'
    unpack_dir = base / archive_hash[:16]
    marker = unpack_dir / '.extraction_complete'
    if not marker.exists():
        unpack_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATA_SOURCE) as archive:
            for member in archive.infolist():
                target = (unpack_dir / member.filename).resolve()
                if not target.is_relative_to(unpack_dir.resolve()):
                    raise ValueError('Unsafe ZIP member path.')
            archive.extractall(unpack_dir)
        marker.write_text(archive_hash)
    roots = [p.parent for p in unpack_dir.rglob('data.yaml') if (p.parent / 'images' / 'train').is_dir()]
else:
    roots = [DATA_SOURCE] if (DATA_SOURCE / 'images' / 'train').is_dir() else [p.parent for p in DATA_SOURCE.rglob('data.yaml') if (p.parent / 'images' / 'train').is_dir()]
assert len(roots) == 1, f'Expected exactly one active dataset root, found {roots}'
DATA_ROOT = roots[0]
extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
records = {}
fingerprint = hashlib.sha256()
summary_rows = []
integrity_file = DATA_ROOT / 'integrity_sha256.json'
integrity = json.loads(integrity_file.read_text()) if integrity_file.exists() else {}
for split in ALL_SPLITS:
    image_dir, label_dir = DATA_ROOT / 'images' / split, DATA_ROOT / 'labels' / split
    assert image_dir.is_dir() and label_dir.is_dir(), f'Missing split: {split}'
    paths = sorted(p for p in image_dir.rglob('*') if p.suffix.lower() in extensions)
    assert paths, f'Empty split: {split}'
    split_items = []
    box_counts = np.zeros(NUM_CLASSES, dtype=int)
    for path in paths:
        rel = path.relative_to(image_dir)
        label = label_dir / rel.with_suffix('.txt')
        assert label.is_file(), f'Missing label: {label}'
        y = np.zeros(NUM_CLASSES, dtype=np.float32)
        for line in label.read_text(encoding='utf-8-sig').splitlines():
            if not line.strip():
                continue
            v = np.asarray([float(x) for x in line.split()])
            assert len(v) == 5 and np.isfinite(v).all(), f'Invalid label: {label}'
            c, x, yy, w, h = v
            assert c == int(c) and 0 <= c < NUM_CLASSES
            assert 0 <= x <= 1 and 0 <= yy <= 1 and 0 < w <= 1 and 0 < h <= 1
            assert min(x-w/2, yy-h/2) >= -1e-4 and max(x+w/2, yy+h/2) <= 1.0001
            y[int(c)] = 1
            box_counts[int(c)] += 1
        for member in [path, label]:
            key = member.relative_to(DATA_ROOT).as_posix()
            digest = file_sha256(member)
            if key in integrity:
                assert digest == integrity[key], f'File changed: {key}'
            fingerprint.update((key + '\0' + digest + '\n').encode())
        split_items.append({'path': str(path), 'image': path.relative_to(DATA_ROOT).as_posix(), 'target': y})
    expected_labels = {p.relative_to(image_dir).with_suffix('.txt') for p in paths}
    assert {p.relative_to(label_dir) for p in label_dir.rglob('*.txt')} == expected_labels, f'Orphan labels in {split}'
    records[split] = split_items
    summary_rows.append({'split': split, 'images': len(paths), 'boxes': int(box_counts.sum())})
DATASET_FINGERPRINT = fingerprint.hexdigest()
expected_counts = {'train': 6404, 'val': 797, 'test': 799, 'saudi_test': 100}
assert {s: len(v) for s, v in records.items()} == expected_counts, 'Dataset partition counts do not match the supplied configuration.'
display(pd.DataFrame(summary_rows))
pd.DataFrame(summary_rows).to_csv(OUT_DIR / 'data_summary.csv', index=False)
print('Dataset:', DATA_ROOT)
print('Dataset fingerprint:', DATASET_FINGERPRINT)


In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class RoadDataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        item = self.items[index]
        with Image.open(item['path']) as image:
            x = transform(image.convert('RGB'))
        return x, torch.from_numpy(item['target'])

generator = torch.Generator().manual_seed(SEED)
loaders = {split: DataLoader(RoadDataset(rows), batch_size=BATCH_SIZE,
    shuffle=(split == 'train'), num_workers=0, generator=generator)
    for split, rows in records.items()}

CHECKPOINT = OUT_DIR / 'baseline_resnet18.pt'
IDENTITY = {
    'architecture': 'resnet18', 'dataset_fingerprint': DATASET_FINGERPRINT,
    'classes': CLASS_NAMES, 'image_size': IMG_SIZE, 'pretrained': PRETRAINED,
    'seed': SEED, 'batch_size': BATCH_SIZE, 'optimizer': 'SGD',
    'learning_rate': LEARNING_RATE, 'momentum': 0.0, 'weight_decay': 0.0,
    'loss': 'BCEWithLogitsLoss_unweighted', 'augmentation': False,
    'checkpoint_selection': 'last_epoch', 'classification_threshold': THRESHOLD,
}
saved = torch.load(CHECKPOINT, map_location='cpu', weights_only=False) if CHECKPOINT.exists() else None
if saved is not None:
    assert saved.get('identity') == IDENTITY, 'This output directory belongs to different data/settings. Use a new OUT_DIR; do not bypass this check.'
clf = models.resnet18(weights=None)
if saved is None and PRETRAINED:
    if IMAGENET_WEIGHTS_PATH:
        pretrained_state = torch.load(IMAGENET_WEIGHTS_PATH, map_location='cpu', weights_only=True)
    else:
        pretrained_state = models.ResNet18_Weights.IMAGENET1K_V1.get_state_dict(progress=True)
    clf.load_state_dict(pretrained_state)
clf.fc = nn.Linear(clf.fc.in_features, NUM_CLASSES)
if saved is not None:
    clf.load_state_dict(saved['state_dict'])
clf = clf.to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(clf.parameters(), lr=LEARNING_RATE, momentum=0, weight_decay=0)
if saved is not None:
    optimizer.load_state_dict(saved['optimizer'])
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(DEVICE)
print('Starting point:', 'saved checkpoint' if saved is not None else 'ImageNet' if PRETRAINED else 'random initialization')


In [ ]:
@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    probabilities, targets = [], []
    for x, y in loader:
        probabilities.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())
        targets.append(y.numpy())
    return np.concatenate(probabilities), np.concatenate(targets)

def classification_macro_ap(P, Y):
    values = [average_precision_score(Y[:, c], P[:, c])
              for c in range(NUM_CLASSES) if Y[:, c].sum() > 0]
    return float(np.mean(values)) if values else float('nan')

def train_or_load():
    checkpoint = torch.load(CHECKPOINT, map_location='cpu', weights_only=False) if CHECKPOINT.exists() else None
    history = checkpoint['history'] if checkpoint else []
    completed = int(checkpoint['epoch']) if checkpoint else 0
    if checkpoint:
        assert checkpoint['identity'] == IDENTITY
        clf.load_state_dict(checkpoint['state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        generator.set_state(checkpoint['loader_rng'])
        torch.set_rng_state(checkpoint['torch_rng'])
        if DEVICE.type == 'cuda' and checkpoint.get('cuda_rng') is not None:
            if len(checkpoint['cuda_rng']) == torch.cuda.device_count():
                torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
    if MODE == 'load':
        assert checkpoint is not None, f'No saved weights: {CHECKPOINT}'
        print(f'Loaded epoch {completed}. Training was not started.')
        return history, completed
    if completed >= EPOCHS:
        print(f'Already trained {completed}/{EPOCHS} epochs. Training skipped.')
        return history, completed
    for epoch in range(completed + 1, EPOCHS + 1):
        clf.train()
        total_loss = 0.0
        for x, y in loaders['train']:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(clf(x), y)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * len(x)
        Pv, Yv = predict_probs(clf, loaders['val'])
        row = {'epoch': epoch, 'train_loss': total_loss / len(records['train']),
               'val_classification_macro_AP': classification_macro_ap(Pv, Yv)}
        history.append(row)
        payload = {'state_dict': {k: v.detach().cpu().clone() for k, v in clf.state_dict().items()},
            'optimizer': optimizer.state_dict(), 'epoch': epoch, 'history': history,
            'identity': IDENTITY, 'loader_rng': generator.get_state(),
            'torch_rng': torch.get_rng_state(),
            'cuda_rng': torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else None}
        temporary = CHECKPOINT.with_suffix('.tmp')
        torch.save(payload, temporary)
        os.replace(temporary, CHECKPOINT)
        pd.DataFrame(history).to_csv(OUT_DIR / 'training_history.csv', index=False)
        print(f"Epoch {epoch}/{EPOCHS} | loss={row['train_loss']:.4f} | val classification AP={row['val_classification_macro_AP']:.4f}")
    return history, EPOCHS

history, COMPLETED_EPOCHS = train_or_load()
display(pd.DataFrame(history))


In [ ]:
def evaluate_image_scores(P, Y, split, model_name):
    P, Y = np.asarray(P), np.asarray(Y)
    assert P.shape == Y.shape and P.shape[1] == NUM_CLASSES
    assert np.isfinite(P).all() and ((P >= 0) & (P <= 1)).all()
    assert np.isin(Y, [0, 1]).all()
    predictions = P >= THRESHOLD
    rows = []
    for c, name in enumerate(CLASS_NAMES):
        precision, recall, f1, _ = precision_recall_fscore_support(
            Y[:, c], predictions[:, c], average='binary', zero_division=0)
        ap = average_precision_score(Y[:, c], P[:, c]) if Y[:, c].sum() else float('nan')
        rows.append({'model': model_name, 'split': split, 'class': name,
            'images': len(Y), 'positive_images': int(Y[:, c].sum()),
            'classification_precision': precision, 'classification_recall': recall,
            'classification_F1': f1, 'classification_AP': ap})
    result = pd.DataFrame(rows)
    result.loc[len(result)] = {'model': model_name, 'split': split, 'class': 'MACRO',
        'images': len(Y), 'positive_images': int(Y.sum()),
        **{column: result[column].mean() for column in
           ['classification_precision', 'classification_recall', 'classification_F1', 'classification_AP']}}
    result['image_exact_match'] = float(np.all(predictions == Y, axis=1).mean())
    return result

protocol = {'dataset_fingerprint': DATASET_FINGERPRINT, 'classes': CLASS_NAMES,
    'threshold': THRESHOLD, 'task': 'image_level_multilabel_classification',
    'detector_score_rule': 'maximum raw box confidence for each class per image; zero when absent',
    'test_images': {s: [r['image'] for r in records[s]] for s in TEST_SPLITS}}
(OUT_DIR / 'protocol.json').write_text(json.dumps(protocol, indent=2), encoding='utf-8')
results = []
for split in TEST_SPLITS:
    P, Y = predict_probs(clf, loaders[split])
    results.append(evaluate_image_scores(P, Y, split, 'ResNet18_plain'))
    predictions = pd.DataFrame({'image': [r['image'] for r in records[split]]})
    for c, name in enumerate(CLASS_NAMES):
        predictions[f'true_{name}'] = Y[:, c].astype(int)
        predictions[f'prob_{name}'] = P[:, c]
    predictions.to_csv(OUT_DIR / f'predictions_{split}.csv', index=False)
baseline_results = pd.concat(results, ignore_index=True)
baseline_results.to_csv(OUT_DIR / 'baseline_classification_results.csv', index=False)
display(baseline_results.round(4))
print('Completed epochs:', COMPLETED_EPOCHS, '| Fixed classification threshold:', THRESHOLD)
print('These AP values are classification AP, NOT detection mAP@50.')


In [ ]:
if MAIN_MODEL_PREDICTIONS_DIR:
    main_dir = Path(MAIN_MODEL_PREDICTIONS_DIR)
    main_protocol = json.loads((main_dir / 'protocol.json').read_text(encoding='utf-8'))
    for key in ['dataset_fingerprint', 'classes', 'task', 'test_images']:
        assert main_protocol.get(key) == protocol[key], f'Cannot compare: protocol mismatch in {key}'
    compared = [baseline_results]
    for split in TEST_SPLITS:
        frame = pd.read_csv(main_dir / f'predictions_{split}.csv')
        image_ids = protocol['test_images'][split]
        assert not frame.image.duplicated().any()
        assert set(frame.image) == set(image_ids), 'Different or missing test images.'
        frame = frame.set_index('image').loc[image_ids]
        P = frame[[f'prob_{c}' for c in CLASS_NAMES]].to_numpy(dtype=float)
        Y = frame[[f'true_{c}' for c in CLASS_NAMES]].to_numpy(dtype=float)
        expected_y = np.stack([r['target'] for r in records[split]])
        assert np.array_equal(Y, expected_y), 'Different ground-truth labels.'
        compared.append(evaluate_image_scores(P, Y, split, 'main_model'))
    fair_comparison = pd.concat(compared, ignore_index=True)
    fair_comparison.to_csv(OUT_DIR / 'fair_classification_comparison.csv', index=False)
    display(fair_comparison[fair_comparison['class'] == 'MACRO'].round(4))
else:
    print('Baseline saved. Main-model scores were not supplied; no cross-task comparison was made.')
print('Results:', OUT_DIR)
